# 009 – LoRA Evaluation

Evaluate the fine-tuned Qwen2.5-1.5B LoRA model on PAN2025 val, HC3, and RAID.
Computes all 5 PAN metrics and prints a comparison table against baselines.

## 1. Setup

In [ ]:
# Install dependencies (run once)
# !pip install -q transformers adapters datasets torch scikit-learn accelerate

In [ ]:
import sys
import json
import os
import torch
from pathlib import Path

SRC_DIR = Path("../../src/009-lora-replication")
sys.path.insert(0, str(SRC_DIR))

from config import VAL_FILE, HC3_FILE, RAID_FILE, ARTIFACTS_DIR
from data import load_jsonl, get_tokenizer
from model import load_trained_model
from evaluate import evaluate_dataset, print_comparison

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 2. Load Model

In [ ]:
# Point to your saved adapter (local path or HuggingFace repo)
adapter_path = str(ARTIFACTS_DIR / "lora_adapter")
print(f"Loading adapter from: {adapter_path}")

model = load_trained_model(adapter_path, device=device)
tokenizer = get_tokenizer()

## 3. Evaluate Datasets

In [ ]:
datasets = [
    ("PAN2025 Validation", str(VAL_FILE), "pan2025_val"),
    ("HC3 Wiki",           str(HC3_FILE),  "hc3"),
    ("RAID",               str(RAID_FILE), "raid"),
]

all_results = {}

for name, path, key in datasets:
    if not os.path.exists(path):
        print(f"Skipping {name}: not found at {path}")
        continue

    print(f"\n{'='*50}")
    print(f"Evaluating: {name}")
    print(f"{'='*50}")

    df = load_jsonl(path)
    metrics, probs = evaluate_dataset(model, tokenizer, df,
                                      device=device, desc=name)

    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

    # Save per-dataset results
    with open(ARTIFACTS_DIR / f"{key}_results.json", "w") as f:
        json.dump(metrics, f, indent=2)
    with open(ARTIFACTS_DIR / f"{key}_predictions.json", "w") as f:
        json.dump({"probabilities": probs.tolist()}, f)

    all_results[name] = metrics

# Save combined
with open(ARTIFACTS_DIR / "all_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nResults saved to {ARTIFACTS_DIR}")

## 4. Comparison Table

In [ ]:
print_comparison(all_results)

## 5. Full Metrics Table

In [ ]:
print(f"{'Dataset':<22} {'ROC-AUC':>10} {'Brier':>10} {'F1':>10} {'C@1':>10} {'F0.5u':>10}")
print("-" * 72)
for name, metrics in all_results.items():
    print(f"{name:<22} {metrics.get('ROC-AUC',0):>10.4f} {metrics.get('Brier',0):>10.4f} "
          f"{metrics.get('F1',0):>10.4f} {metrics.get('C@1',0):>10.4f} {metrics.get('F0.5u',0):>10.4f}")